In [20]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
%pip install prophet
from statsmodels.tsa.stattools       import adfuller, grangercausalitytests
from statsmodels.tsa.seasonal        import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters     import ExponentialSmoothing
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.graphics.tsaplots   import plot_acf, plot_pacf
from sklearn.preprocessing           import MinMaxScaler
from sklearn.metrics                 import mean_absolute_error, mean_squared_error
from prophet                         import Prophet


In [24]:
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    na_values='?',
    low_memory=False
)
# Combine Date + Time into a single Datetime index
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df = df.drop(columns=['Date', 'Time'])
df = df.set_index('Datetime').sort_index()

# Drop rows where target is missing
df = df.dropna(subset=['Global_active_power'])

# Forward-fill remaining missing values in exogenous columns
df = df.ffill()

print(f"Raw data shape   : {df.shape}")
print(f"Date range       : {df.index[0]} → {df.index[-1]}")
print(f"Columns          : {df.columns.tolist()}")

# ── Resample to DAILY means (minute-level → daily) ───────────────────────────
daily = df.resample('D').mean().dropna()
print(f"\nDaily data shape : {daily.shape}")
print(daily.head())

# Target and exogenous feature lists
TARGET  = 'Global_active_power'
EXOG_COLS = ['Global_reactive_power', 'Voltage',
             'Global_intensity', 'Sub_metering_1',
             'Sub_metering_2', 'Sub_metering_3']

y    = daily[TARGET]
exog = daily[EXOG_COLS]



Raw data shape   : (2049280, 7)
Date range       : 2006-12-16 17:24:00 → 2010-11-26 21:02:00
Columns          : ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

Daily data shape : (1433, 7)
            Global_active_power  Global_reactive_power     Voltage  \
Datetime                                                             
2006-12-16             3.053475               0.088187  236.243763   
2006-12-17             2.354486               0.156949  240.087028   
2006-12-18             1.530435               0.112356  241.231694   
2006-12-19             1.157079               0.104821  241.999313   
2006-12-20             1.545658               0.111804  242.308062   

            Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
Datetime                                                                      
2006-12-16         13.082828        0.000000        1.378788       12.43939

In [25]:
# ── Fig 1 : Target time series overview ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
ax.plot(y.index, y.values, color='#2563eb', linewidth=0.9)
ax.set_title('Daily Mean – Global Active Power', fontsize=14, fontweight='bold')
ax.set_ylabel('kW')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.savefig('fig1_target_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nFig1 saved.")

# ── Fig 2 : Correlation heatmap (all features vs target) ─────────────────────
fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
corr = daily.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 10})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
fig.savefig('fig2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig2 saved.")

# ── Fig 3 : All features plotted together ────────────────────────────────────
fig, axes = plt.subplots(len(daily.columns), 1,
                         figsize=(16, 3 * len(daily.columns)),
                         constrained_layout=True)
fig.suptitle('All Features Over Time', fontsize=15, fontweight='bold')
colors = ['#2563eb','#dc2626','#16a34a','#d97706','#9333ea','#0891b2','#f43f5e']
for ax, col, c in zip(axes, daily.columns, colors):
    ax.plot(daily.index, daily[col].values, color=c, linewidth=0.8)
    ax.set_title(col, fontsize=11)
    ax.set_ylabel(col)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.savefig('fig3_all_features.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig3 saved.")




Fig1 saved.
Fig2 saved.
Fig3 saved.


In [26]:
print("\n--- ADF Stationarity Tests ---")
for col in daily.columns:
    result = adfuller(daily[col].dropna())
    status = 'STATIONARY' if result[1] < 0.05 else 'NON-STATIONARY'
    print(f"  {col:<30} p={result[1]:.5f}  →  {status}")



--- ADF Stationarity Tests ---
  Global_active_power            p=0.00434  →  STATIONARY
  Global_reactive_power          p=0.00309  →  STATIONARY
  Voltage                        p=0.00332  →  STATIONARY
  Global_intensity               p=0.00309  →  STATIONARY
  Sub_metering_1                 p=0.00000  →  STATIONARY
  Sub_metering_2                 p=0.00001  →  STATIONARY
  Sub_metering_3                 p=0.00140  →  STATIONARY


In [27]:
print("\n--- Granger Causality Tests (does X help predict Global_active_power?) ---")
gc_data = daily[[TARGET] + EXOG_COLS].dropna()
for col in EXOG_COLS:
    test_df = gc_data[[TARGET, col]]
    result  = grangercausalitytests(test_df, maxlag=7, verbose=False)
    # Use minimum p-value across lags
    min_p = min([result[lag][0]['ssr_ftest'][1] for lag in range(1, 8)])
    useful = 'USEFUL' if min_p < 0.05 else 'NOT USEFUL'
    print(f"  {col:<30} min_p={min_p:.5f}  →  {useful}")



--- Granger Causality Tests (does X help predict Global_active_power?) ---
  Global_reactive_power          min_p=0.00000  →  USEFUL
  Voltage                        min_p=0.00000  →  USEFUL
  Global_intensity               min_p=0.00000  →  USEFUL
  Sub_metering_1                 min_p=0.00000  →  USEFUL
  Sub_metering_2                 min_p=0.00000  →  USEFUL
  Sub_metering_3                 min_p=0.00004  →  USEFUL


In [28]:
decomp = seasonal_decompose(y, model='additive', period=7,
                            extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Seasonal Decomposition of Global Active Power (period=7)',
             fontsize=14, fontweight='bold')
components = [('Observed', y, '#2563eb'),
              ('Trend',    decomp.trend,    '#dc2626'),
              ('Seasonal', decomp.seasonal, '#16a34a'),
              ('Residual', decomp.resid,    '#9333ea')]
for ax, (title, data, col) in zip(axes, components):
    ax.plot(data.index, data.values, color=col, linewidth=0.9)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('kW')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.savefig('fig4_decomposition.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nFig4 saved.")



Fig4 saved.


In [29]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)
fig.suptitle('ACF & PACF – Global Active Power', fontsize=14, fontweight='bold')
plot_acf(y,  lags=60, ax=axes[0], color='#2563eb')
axes[0].set_title('Autocorrelation Function (ACF)', fontsize=12)
plot_pacf(y, lags=60, ax=axes[1], color='#dc2626', method='ywm')
axes[1].set_title('Partial Autocorrelation Function (PACF)', fontsize=12)
fig.savefig('fig5_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig5 saved.")


Fig5 saved.


In [30]:
TEST_DAYS = 60

y_train    = y[:-TEST_DAYS]
y_test     = y[-TEST_DAYS:]
exog_train = exog[:-TEST_DAYS]
exog_test  = exog[-TEST_DAYS:]

print(f"\nTrain: {len(y_train)} days  |  Test: {len(y_test)} days")
print(f"Test period: {y_test.index[0].date()} → {y_test.index[-1].date()}")



Train: 1373 days  |  Test: 60 days
Test period: 2010-09-28 → 2010-11-26


In [31]:
print("\nFitting SARIMA...")
sarima_model = SARIMAX(y_train,
                       order=(1, 1, 1),
                       seasonal_order=(1, 1, 1, 7),
                       enforce_stationarity=False,
                       enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)
sarima_fc  = sarima_fit.forecast(steps=TEST_DAYS)

sarima_mae  = mean_absolute_error(y_test, sarima_fc)
sarima_rmse = np.sqrt(mean_squared_error(y_test, sarima_fc))
sarima_mape = np.mean(np.abs((y_test.values - sarima_fc.values) / y_test.values)) * 100
print(f"  MAE={sarima_mae:.4f}  RMSE={sarima_rmse:.4f}  MAPE={sarima_mape:.2f}%")



Fitting SARIMA...
  MAE=0.3814  RMSE=0.4479  MAPE=29.82%


In [32]:
print("Fitting SARIMAX...")

# Scale exogenous features (helps SARIMAX convergence)
scaler     = MinMaxScaler()
exog_train_sc = scaler.fit_transform(exog_train)
exog_test_sc  = scaler.transform(exog_test)

sarimax_model = SARIMAX(y_train,
                        exog=exog_train_sc,
                        order=(1, 1, 1),
                        seasonal_order=(1, 1, 1, 7),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
sarimax_fit = sarimax_model.fit(disp=False)
sarimax_fc  = sarimax_fit.forecast(steps=TEST_DAYS, exog=exog_test_sc)

sarimax_mae  = mean_absolute_error(y_test, sarimax_fc)
sarimax_rmse = np.sqrt(mean_squared_error(y_test, sarimax_fc))
sarimax_mape = np.mean(np.abs((y_test.values - sarimax_fc.values) / y_test.values)) * 100
print(f"  MAE={sarimax_mae:.4f}  RMSE={sarimax_rmse:.4f}  MAPE={sarimax_mape:.2f}%")


Fitting SARIMAX...
  MAE=0.0072  RMSE=0.0086  MAPE=0.62%


In [33]:
print("Fitting Holt-Winters...")
hw_model = ExponentialSmoothing(y_train,
                                trend='add',
                                seasonal='add',
                                seasonal_periods=7,
                                initialization_method='estimated')
hw_fit = hw_model.fit()
hw_fc  = hw_fit.forecast(steps=TEST_DAYS)

hw_mae  = mean_absolute_error(y_test, hw_fc)
hw_rmse = np.sqrt(mean_squared_error(y_test, hw_fc))
hw_mape = np.mean(np.abs((y_test.values - hw_fc.values) / y_test.values)) * 100
print(f"  MAE={hw_mae:.4f}  RMSE={hw_rmse:.4f}  MAPE={hw_mape:.2f}%")


Fitting Holt-Winters...
  MAE=0.4244  RMSE=0.4869  MAPE=33.31%


In [34]:
print("Fitting VAR...")
var_data  = daily[[TARGET] + EXOG_COLS].dropna()
var_train = var_data[:-TEST_DAYS]
var_test  = var_data[-TEST_DAYS:]

var_model = VAR(var_train)
var_fit   = var_model.fit(maxlags=7, ic='aic')
print(f"  VAR selected lag order: {var_fit.k_ar}")

# Forecast using last k_ar rows as the lagged input
lag_order  = var_fit.k_ar
var_input  = var_train.values[-lag_order:]
var_fc_raw = var_fit.forecast(y=var_input, steps=TEST_DAYS)
var_fc     = pd.Series(var_fc_raw[:, 0], index=y_test.index)

var_mae  = mean_absolute_error(y_test, var_fc)
var_rmse = np.sqrt(mean_squared_error(y_test, var_fc))
var_mape = np.mean(np.abs((y_test.values - var_fc.values) / y_test.values)) * 100
print(f"  MAE={var_mae:.4f}  RMSE={var_rmse:.4f}  MAPE={var_mape:.2f}%")


Fitting VAR...
  VAR selected lag order: 7
  MAE=0.3001  RMSE=0.3692  MAPE=23.77%


In [35]:
fig, axes = plt.subplots(4, 1, figsize=(16, 18), constrained_layout=True)
fig.suptitle('Classical Model Forecasts vs Actuals (60-Day Horizon)',
             fontsize=15, fontweight='bold')

model_results = {
    'SARIMA (1,1,1)(1,1,1)7 — Univariate Baseline':
        (sarima_fc,  sarima_mae,  sarima_rmse,  sarima_mape,  '#dc2626'),
    'SARIMAX — SARIMA + Exogenous Features':
        (sarimax_fc, sarimax_mae, sarimax_rmse, sarimax_mape, '#2563eb'),
    'Holt-Winters (Triple Exponential Smoothing)':
        (hw_fc,      hw_mae,      hw_rmse,      hw_mape,      '#16a34a'),
    'VAR (Vector Autoregression) — Fully Multivariate':
        (var_fc,     var_mae,     var_rmse,      var_mape,     '#d97706'),
}

for ax, (name, (fc, mae, rmse, mape, col)) in zip(axes, model_results.items()):
    ax.plot(y_train[-90:].index, y_train[-90:].values,
            color='#374151', linewidth=1.1, label='Train (last 90d)')
    ax.plot(y_test.index, y_test.values,
            color='#6b7280', linewidth=1.5, linestyle='--', label='Actual')
    ax.plot(y_test.index, fc.values,
            color=col, linewidth=1.8,
            label=f'Forecast  |  MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:.1f}%')
    ax.axvline(y_test.index[0], color='k', linestyle=':', linewidth=1)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('kW')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.savefig('fig6_classical_forecasts.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nFig6 saved.")



Fig6 saved.


In [36]:
print("\nFitting Prophet with exogenous regressors...")

# Prophet requires: ds (datetime), y (target), + regressor columns
prophet_df = daily[[TARGET] + EXOG_COLS].reset_index()
prophet_df = prophet_df.rename(columns={'Datetime': 'ds', TARGET: 'y'})

# Scale exogenous features (Prophet prefers normalized regressors)
for col in EXOG_COLS:
    col_mean = prophet_df[col][:-TEST_DAYS].mean()
    col_std  = prophet_df[col][:-TEST_DAYS].std()
    prophet_df[col] = (prophet_df[col] - col_mean) / (col_std + 1e-8)

prophet_train_df = prophet_df.iloc[:-TEST_DAYS]
prophet_test_df  = prophet_df.iloc[-TEST_DAYS:]

# Build Prophet model and add each exogenous column as a regressor
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95,
    seasonality_mode='additive',
    changepoint_prior_scale=0.05   # controls trend flexibility
)
for col in EXOG_COLS:
    m.add_regressor(col)

m.fit(prophet_train_df)

# Future dataframe must include regressor values for the forecast period
future = pd.concat([prophet_train_df, prophet_test_df], ignore_index=True)
forecast = m.predict(future)

# Extract only the test-period predictions
prophet_fc = forecast.iloc[-TEST_DAYS:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
prophet_fc = prophet_fc.set_index('ds')

prophet_mae  = mean_absolute_error(y_test.values, prophet_fc['yhat'].values)
prophet_rmse = np.sqrt(mean_squared_error(y_test.values, prophet_fc['yhat'].values))
prophet_mape = np.mean(np.abs((y_test.values - prophet_fc['yhat'].values) / y_test.values)) * 100
print(f"  MAE={prophet_mae:.4f}  RMSE={prophet_rmse:.4f}  MAPE={prophet_mape:.2f}%")



Fitting Prophet with exogenous regressors...


13:35:27 - cmdstanpy - INFO - Chain [1] start processing
13:35:31 - cmdstanpy - INFO - Chain [1] done processing


  MAE=0.0052  RMSE=0.0068  MAPE=0.44%


In [37]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), constrained_layout=True)
fig.suptitle('Prophet Forecast with Exogenous Regressors',
             fontsize=15, fontweight='bold')

# Full range
ax = axes[0]
ax.plot(y.index, y.values, color='#374151', linewidth=0.8, label='Observed')
ax.plot(forecast['ds'], forecast['yhat'],
        color='#dc2626', linewidth=1.1, label='Forecast')
ax.fill_between(forecast['ds'],
                forecast['yhat_lower'], forecast['yhat_upper'],
                alpha=0.2, color='#dc2626', label='95% CI')
ax.axvline(y_test.index[0], color='k', linestyle=':', linewidth=1.2,
           label='Train/Test Split')
ax.set_title('Full Time Range', fontsize=12)
ax.set_ylabel('kW'); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Zoom into test period
ax2 = axes[1]
plot_start = y_train.index[-90]
zoom_y     = y[y.index >= plot_start]
zoom_fc    = forecast[forecast['ds'] >= plot_start]
ax2.plot(zoom_y.index, zoom_y.values,
         color='#374151', linewidth=1.3, label='Observed')
ax2.plot(zoom_fc['ds'], zoom_fc['yhat'],
         color='#dc2626', linewidth=1.7,
         label=f'Forecast  |  MAE={prophet_mae:.3f}  RMSE={prophet_rmse:.3f}  MAPE={prophet_mape:.1f}%')
ax2.fill_between(zoom_fc['ds'],
                 zoom_fc['yhat_lower'], zoom_fc['yhat_upper'],
                 alpha=0.2, color='#dc2626')
ax2.axvline(y_test.index[0], color='k', linestyle=':', linewidth=1.2)
ax2.set_title('Zoom: Last 90 Training Days + 60-Day Forecast', fontsize=12)
ax2.set_ylabel('kW'); ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.savefig('fig7_prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig7 saved.")

# ── Fig 8 : Prophet components ────────────────────────────────────────────────
comp_fig = m.plot_components(forecast)
comp_fig.set_size_inches(14, 12)
comp_fig.suptitle('Prophet – Trend & Seasonality Components',
                  fontsize=14, fontweight='bold', y=1.01)
comp_fig.tight_layout()
comp_fig.savefig('fig8_prophet_components.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig8 saved.")


Fig7 saved.
Fig8 saved.


In [38]:
model_names = ['SARIMA\n(Univariate)', 'SARIMAX\n(+Exog)', 'Holt-Winters', 'VAR', 'Prophet\n(+Regressors)']
all_mae  = [sarima_mae,  sarimax_mae,  hw_mae,  var_mae,  prophet_mae]
all_rmse = [sarima_rmse, sarimax_rmse, hw_rmse, var_rmse, prophet_rmse]
all_mape = [sarima_mape, sarimax_mape, hw_mape, var_mape, prophet_mape]

x = np.arange(len(model_names))
w = 0.25
bar_colors = ['#2563eb', '#dc2626', '#16a34a']

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
b1 = ax.bar(x - w,   all_mae,  w, label='MAE',  color=bar_colors[0], alpha=0.85)
b2 = ax.bar(x,       all_rmse, w, label='RMSE', color=bar_colors[1], alpha=0.85)
b3 = ax.bar(x + w,   all_mape, w, label='MAPE (%)', color=bar_colors[2], alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Error')
ax.set_title('Model Comparison – MAE, RMSE, MAPE on 60-Day Test Set',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, max(all_mape) * 1.3)

fig.savefig('fig9_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig9 saved.")


Fig9 saved.


In [39]:
print("\n" + "="*65)
print("  MODEL PERFORMANCE SUMMARY  (60-Day Forecast Horizon)")
print("="*65)
print(f"  {'Model':<32} {'MAE':>7} {'RMSE':>7} {'MAPE':>8}")
print("-"*65)
for name, mae, rmse, mape in zip(model_names, all_mae, all_rmse, all_mape):
    clean = name.replace('\n', ' ')
    print(f"  {clean:<32} {mae:>7.4f} {rmse:>7.4f} {mape:>7.2f}%")
print("="*65)
best = model_names[np.argmin(all_rmse)].replace('\n', ' ')
print(f"\n  Best Model (lowest RMSE) → {best}")
print("="*65)



  MODEL PERFORMANCE SUMMARY  (60-Day Forecast Horizon)
  Model                                MAE    RMSE     MAPE
-----------------------------------------------------------------
  SARIMA (Univariate)               0.3814  0.4479   29.82%
  SARIMAX (+Exog)                   0.0072  0.0086    0.62%
  Holt-Winters                      0.4244  0.4869   33.31%
  VAR                               0.3001  0.3692   23.77%
  Prophet (+Regressors)             0.0052  0.0068    0.44%

  Best Model (lowest RMSE) → Prophet (+Regressors)
